# 📊 01 - Exploratory Data Analysis (EDA)

**Credit Card Fraud Detection — ULB Dataset**

This notebook performs comprehensive EDA on the Kaggle credit card fraud dataset:
- Class imbalance analysis
- Feature distributions by class
- Correlation analysis
- PCA separability (t-SNE/UMAP)
- Business cost framing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Optional: t-SNE / UMAP
from sklearn.manifold import TSNE
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print('UMAP not installed. Will use t-SNE only.')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded.')

## 1. Load Data

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from src.data_loader import DataLoader

loader = DataLoader('Dataset/Dataset/creditcard.csv')
df = loader.load()
stats = loader.get_basic_stats()

print(f'\nDataset shape: {df.shape}')
print(f'\nStatistics:')
for k, v in stats.items():
    print(f'  {k}: {v}')

df.head()

## 2. Class Imbalance Analysis

**Key finding:** Only 0.172% of transactions are fraudulent (492 out of 284,807). This extreme imbalance means:
- Accuracy is a **misleading** metric (99.8% accuracy by predicting all legitimate)
- We must use **Precision-Recall AUC** as our primary metric
- Business framing: each missed fraud costs ~$150, each review costs ~$5

In [ ]:
# Class distribution
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
ax1 = axes[0]
bars = ax1.bar(['Legitimate', 'Fraud'], class_counts.values, color=colors, edgecolor='white', linewidth=2)
for bar, count, pct in zip(bars, class_counts.values, class_pct.values):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2000,
             f'{count:,}\n({pct:.3f}%)', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax1.set_ylabel('Number of Transactions', fontsize=12)
ax1.set_title('Class Distribution — Extreme Imbalance', fontsize=14, fontweight='bold')

# Pie chart (zoomed in on fraud)
ax2 = axes[1]
ax2.pie([class_counts[0], class_counts[1]],
        labels=['Legitimate', 'Fraud'],
        colors=colors,
        autopct='%1.4f%%',
        startangle=90,
        explode=(0, 0.2),
        textprops={'fontsize': 12})
ax2.set_title(f'Fraud is only {class_pct[1]:.3f}% of all transactions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('data/processed/class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

# Business impact
AVG_FRAUD_LOSS = 150  # dollars per missed fraud
REVIEW_COST = 5       # dollars per flagged transaction
n_fraud = int(class_counts[1])
baseline_loss = n_fraud * AVG_FRAUD_LOSS
print(f'\n💰 BUSINESS IMPACT (Baseline — no detection):')
print(f'   {n_fraud} fraudulent transactions × ${AVG_FRAUD_LOSS} = ${baseline_loss:,.0f} in losses')
print(f'   Even catching 80% saves ${baseline_loss * 0.8:,.0f}!')

## 3. Feature Distributions by Class

Key features (Amount, Time, top V features) often show distinct patterns between fraud and legit.

In [ ]:
# Amount distribution by class
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Amount histogram
ax = axes[0]
for cls, color, label in [(0, '#2ecc71', 'Legitimate'), (1, '#e74c3c', 'Fraud')]:
    subset = df[df['Class'] == cls]['Amount']
    ax.hist(subset, bins=100, alpha=0.6, color=color, label=label, density=True, range=(0, 500))
ax.set_xlabel('Amount ($)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Transaction Amount by Class', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

# Time histogram
ax = axes[1]
for cls, color, label in [(0, '#2ecc71', 'Legitimate'), (1, '#e74c3c', 'Fraud')]:
    subset = df[df['Class'] == cls]['Time'] / 3600  # Convert to hours
    ax.hist(subset, bins=100, alpha=0.6, color=color, label=label, density=True)
ax.set_xlabel('Time (hours)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Transaction Time by Class', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

# Log Amount
ax = axes[2]
df['LogAmount'] = np.log1p(df['Amount'])
for cls, color, label in [(0, '#2ecc71', 'Legitimate'), (1, '#e74c3c', 'Fraud')]:
    subset = df[df['Class'] == cls]['LogAmount']
    ax.hist(subset, bins=100, alpha=0.6, color=color, label=label, density=True)
ax.set_xlabel('Log(Amount + 1)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Log-Transformed Amount by Class', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('data/processed/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top V-features by class (box plots)
top_features = ['V14', 'V4', 'V12', 'V10', 'V17', 'V16', 'V3', 'V11']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, feat in enumerate(top_features):
    ax = axes[i // 4][i % 4]
    data_0 = df[df['Class'] == 0][feat]
    data_1 = df[df['Class'] == 1][feat]
    
    ax.boxplot([data_0.values, data_1.values],
               labels=['Legit', 'Fraud'],
               patch_artist=True,
               boxprops=[dict(facecolor='#2ecc71', alpha=0.5), dict(facecolor='#e74c3c', alpha=0.5)])
    ax.set_title(feat, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Top Discriminative PCA Features by Class', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('data/processed/feature_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation heatmap
corr_cols = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Time']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('data/processed/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation with target
target_corr = df[corr_cols + ['Class']].corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)
print('\nTop features correlated with fraud (Class):')
print(target_corr.head(10))

## 5. PCA Separability — t-SNE Visualization

Visualize the 28 PCA features in 2D to see if fraud clusters are separable.

In [ ]:
# Subsample for t-SNE (it's expensive)
np.random.seed(42)
n_sample = 10000
fraud_idx = df[df['Class'] == 1].index
legit_idx = df[df['Class'] == 0].sample(n_sample - len(fraud_idx), random_state=42).index
sample_idx = np.concatenate([fraud_idx, legit_idx])

X_sample = df.loc[sample_idx, [f'V{i}' for i in range(1, 29)] + ['Amount']]
y_sample = df.loc[sample_idx, 'Class']

# t-SNE
print('Running t-SNE (this may take a minute)...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_sample)

fig, axes = plt.subplots(1, 2 if HAS_UMAP else 1, figsize=(20 if HAS_UMAP else 10, 8))
if not HAS_UMAP:
    axes = [axes]

# t-SNE plot
ax = axes[0]
for cls, color, label, marker in [(0, '#2ecc71', 'Legitimate', 'o'), (1, '#e74c3c', 'Fraud', 'X')]:
    mask = y_sample == cls
    alpha = 0.3 if cls == 0 else 0.9
    size = 10 if cls == 0 else 100
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               c=color, label=label, alpha=alpha, s=size, marker=marker, edgecolors='white')
ax.set_title('t-SNE Projection of PCA Features', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(fontsize=12)

# UMAP (if available)
if HAS_UMAP:
    print('Running UMAP...')
    reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = reducer.fit_transform(X_sample)
    
    ax = axes[1]
    for cls, color, label, marker in [(0, '#2ecc71', 'Legitimate', 'o'), (1, '#e74c3c', 'Fraud', 'X')]:
        mask = y_sample == cls
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
                   c=color, label=label, alpha=0.3 if cls == 0 else 0.9,
                   s=10 if cls == 0 else 100, marker=marker, edgecolors='white')
    ax.set_title('UMAP Projection of PCA Features', fontsize=14, fontweight='bold')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('data/processed/pca_tsne_umap.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nNote: Fraud transactions (red X) form small clusters within the legitimate space.')
print('This confirms the need for sophisticated models — linear separation is insufficient.')

## 6. Summary of EDA Findings

| Finding | Implication |
|---------|-------------|
| **Extreme imbalance** (0.172% fraud) | Accuracy is misleading; use PR-AUC |
| **No temporal pattern** in fraud | Time feature has limited predictive power |
| **V14, V4, V12 are most discriminative** | These features drive most fraud detection |
| **Fraud clusters in specific amount ranges** | Amount is a useful but secondary feature |
| **PCA features are uncorrelated** | Good for linear models; tree models can find interactions |
| **t-SNE shows partial separability** | Machine learning models can learn decision boundaries |

**Next step:** Preprocessing with correct train/test split and resampling strategies.